# logit_and_roc

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/logit_and_roc.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们使用 CFPS 成人数据，先剔除就业状态、教育年限、户口类型等变量的缺失/无效值（负值为问卷中的缺失码），然后构造核心变量：被解释变量 exit_labor 表示 2014 年是否退出劳动力市场（employ2014==3），以及年龄、年龄平方项和城镇户口虚拟变量。

In [ ]:
%%stata
clear
set more off
use ../datasets/cfps_adult.dta
// 清洗并产生数据
drop if employ2014<0
drop if te4<0
drop if qa301<0 | qa301==5
gen exit_labor=employ2014==3
gen age=2014-cfps_birth
gen age2=age^2
gen urban_hukou=qa301==3

在正式回归前，我们先用二维列联表（tab）直观地比较不同性别退出劳动力市场的比例，这相当于对几率比（odds ratio）做一个非参数的描述性考察。

In [ ]:
%%stata
// odds ratio
tab exit_labor cfps_gender

我们把解释变量存入局部宏 x 以便重复调用，reportvar 则限定输出表格中展示的系数（省份、教育等控制变量省略显示）。先估计只含性别的 Logit 模型：logit 命令通过极大似然估计 Logistic 分布下的二元选择模型；随后 margins, dydx(*) post 计算所有解释变量的平均边际效应并 posted 到 e(b)，这样 outreg2 报告的就是边际效应而非对数几率比系数，便于直接解释为概率的变化。

In [ ]:
%%stata
// 回归并预测
local x "age age2 cfps_gender urban_hukou i.provcd14 i.te4"
local reportvar "age age2 cfps_gender urban_hukou"
logit exit_labor cfps_gender
margins, dydx(*) post
outreg2 using logit_roc.doc, replace keep(`reportvar') ctitle(Logit)

logistic 命令与 logit 估计的是同一个模型，但默认报告几率比（odds ratio）而非系数，我们把它作为对照列输出，方便比较两种报告方式。

In [ ]:
%%stata
logistic exit_labor cfps_gender
outreg2 using logit_roc.doc, append keep(`reportvar') ctitle(Logistic)

接下来我们加入年龄及其平方项、户口、省份和教育等全套控制变量，分别估计 Logit 与 Probit 模型并报告平均边际效应。Probit 假定潜变量误差服从标准正态分布，与 Logistic 分布形状接近但尾部不同，两者边际效应通常非常接近，可以作为稳健性对照。

In [ ]:
%%stata
logit exit_labor `x'
margins, dydx(*) post
outreg2 using logit_roc.doc, append keep(`reportvar') ctitle(Logit)
probit exit_labor `x'
margins, dydx(*) post
outreg2 using logit_roc.doc, append keep(`reportvar') ctitle(Probit)

再用 logistic 报告完整设定的几率比；最后一列输出后，e() 中保留的就是这个完整 Logit 模型的估计结果，后续预测和拟合优度指标都基于它。

In [ ]:
%%stata
logistic exit_labor `x'
outreg2 using logit_roc.doc, append keep(`reportvar') ctitle(Logistic)

predict 对最近一次估计的模型计算每个样本退出劳动力市场的预测概率 p_exit（即条件概率 Pr(exit=1|x) 的拟合值）。

In [ ]:
%%stata
// 如果用probit模型：probit exit_labor `x'
predict p_exit

我们以 0.5 为分类阈值（cutoff），把预测概率转成 0/1 的分类预测，并生成混淆矩阵的四个元素：真正例 TP、假正例 FP、假负例 FN、真负例 TN，为下面计算查准率、查全率等指标做准备。

In [ ]:
%%stata
// 计算cutoff=0.5时的查准率、查全率
gen predict_exit=p_exit>0.5
gen TP=(exit_labor==1 & predict_exit==1)
gen FP=(exit_labor==0 & predict_exit==1)
gen FN=(exit_labor==1 & predict_exit==0)
gen TN=(exit_labor==0 & predict_exit==0)

循环对每个指示变量求均值：由于 TP 等变量是 0/1 虚拟变量，其样本均值恰好等于对应单元格在总样本中的占比（例如 TP 的均值就是真正例率）。我们把四个占比存入同名局部宏备用。

In [ ]:
%%stata
foreach v of varlist TP FP FN TN{
	quietly: su `v'
	local `v' = r(mean)
}

基于混淆矩阵计算常用分类评价指标：e(r2_p) 是 Logit 的伪 R²（McFadden R²）；查准率（precision）= TP/(TP+FP)，度量预测为正的样本中真正例的比例；查全率（recall，即 sensitivity）= TP/(TP+FN)；精度（accuracy）是整体分类正确的比例；F1 是查准率与查全率的调和平均，在类别不平衡时比精度更有参考价值。

In [ ]:
%%stata
local R2=e(r2_p)
local precision=`TP'/(`TP'+`FP')
local recall=`TP'/(`TP'+`FN')
local accuracy=(`TP'+`TN')/(`TP'+`TN'+`FP'+`FN')
local F1=(2*`precision'*`recall')/(`precision'+`recall')
di "R2=`R2'"
di "查准率=`precision'"
di "查全率=`recall'"
di "精度=`accuracy'"
di "F1=`F1'"

lroc 绘制 ROC 曲线并报告 AUC（曲线下面积）：ROC 曲线以不同阈值下的灵敏度（sensitivity）对 1-特异度（1-specificity）作图，AUC 越接近 1 说明模型的区分能力越强。lsens 则进一步把灵敏度与特异度分别作为阈值的函数画出，便于我们直观地权衡两类错误并选择合适的分类阈值。

In [ ]:
%%stata
// 画出ROC曲线
lroc
// 画出sensitivity 以及specificity
lsens